## Download real datasets

In [1]:
import os, openml, tqdm, json

In [2]:
import numpy as np
import pandas as pd

In [3]:
from pathlib import Path
from openml.tasks import TaskType
from sklearn.preprocessing import LabelEncoder

In [4]:
raw = '../data/raw'
processed = '../data/processed'

## Helper functions

In [5]:
def load_openml(tid):
    task = openml.tasks.get_task(tid)
    X, y, _, _ = task.get_dataset().get_data(target=task.target_name)
    return X, LabelEncoder().fit_transform(y)

In [6]:
def load_split(tid, split):
    blocks = []
    for prefix in ['N', 'C']:
        path = f'{raw}/{tid}/{prefix}_{split}.npy'
        if os.path.exists(path):
            blocks.append(np.load(path, allow_pickle=True))
    X = np.concatenate(blocks, axis=1)
    y = np.load(f'{raw}/{tid}/y_{split}.npy', allow_pickle=True)
    return X, y

In [7]:
def load_talent(tid):
    
    Xs, ys = [], []
    for split in ['train', 'test', 'val']:
        try:
            X, y = load_split(tid, split)
            Xs.append(X)
            ys.append(y)
        except:
            continue
     
    X = np.concatenate(Xs, axis=0)
    y = np.concatenate(ys, axis=0).ravel()
    return X, LabelEncoder().fit_transform(y)

In [8]:
def status_check(suites):
    files = os.listdir(processed)
    return {src:sum(f.startswith(f'{src}__') for f in files) for src in suites}

## Loading CC18, TabArena and Grinsztajn

In [9]:
datasets_openml = {99:'cc18', 457:'tabarena', 334:'grinsztajn'}

In [10]:
for sid, src in datasets_openml.items():
    suite = openml.study.get_suite(sid)
    tasks = openml.tasks.list_tasks(task_id=suite.tasks, output_format='dataframe')
    tasks = tasks[tasks['ttid'] == TaskType.SUPERVISED_CLASSIFICATION]

    tids = tasks['tid'].astype(int).tolist()
    names = tasks['name'].astype(str).tolist()

    for tid, name in tqdm.tqdm(zip(tids, names), total=len(tids)):
        X, y = load_openml(tid)
        X.assign(target=y).to_csv(f'{processed}/{src}__{name}.csv', index=False)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:03<00:00,  2.09it/s]


## Loading Talent

In [11]:
datasets_talent = os.listdir(raw)
datasets_talent[:5]

['Performance-Prediction',
 'Contaminant-detection-in-packaged-cocoa-hazelnut-spread-jars-using-Microwaves-Sensing-and-Machine-Learning-9.5GHz(Urbinati)',
 'philippine',
 'Fitness_Club_c',
 'dry_bean_dataset']

In [12]:
for name in tqdm.tqdm(datasets_talent):
    info = json.loads(Path(f'{raw}/{name}/info.json').read_text())
    if info['task_type'] in ['binclass', 'multiclass']:
        X, y = load_talent(name)
        pd.DataFrame(X).assign(target=y).to_csv(f'{processed}/talent__{name}.csv', index=False)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:16<00:00, 18.67it/s]


## Total datasets

In [13]:
suites = list(datasets_openml.values()) + ['talent']

In [14]:
status_check(suites)

{'cc18': 72, 'tabarena': 38, 'grinsztajn': 7, 'talent': 200}

## Removing duplicates

In [15]:
seen = set()
for file in sorted(os.listdir(processed)):
    name = file.split('__', 1)[1].lower()
    if name in seen:
        os.remove(f'{processed}/{file}')
    else:
        seen.add(name)

In [16]:
status_check(suites)

{'cc18': 72, 'tabarena': 29, 'grinsztajn': 6, 'talent': 147}

## Removing large datasets

In [20]:
for file in sorted(os.listdir(processed)):
    path = f'{processed}/{file}'
    data = pd.read_csv(path)
    if data.shape[0] > 1_000_000 or data.shape[1] > 10_000:
        os.remove(path)

## Final result

In [21]:
status_check(suites)

{'cc18': 72, 'tabarena': 29, 'grinsztajn': 6, 'talent': 147}